In [ ]:
reseñas = df['texto']
etiquetas = df['sentimiento']


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    reseñas,
    etiquetas,
    test_size=0.2, #porcion de pruebas 20%
    random_state=42, #semilla para reproducibilidad 
    stratify=etiquetas #mantener los datos balanceados
)


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer()

X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)


In [ ]:
from sklearn.linear_model import LogisticRegression

modelo = LogisticRegression()

modelo.fit(X_train_vec, y_train)


# Ejercicio 2 — Pipeline completo con reseñas de hoteles

Objetivo: armar desde cero un clasificador de sentimiento (positivo / negativo) para reseñas de hoteles.

Los pasos del pipeline:
```
CSV → split train/test → TF-IDF → LogisticRegression → accuracy → probar con reseñas nuevas
```

## Paso 1 — Cargar el dataset

Leemos el CSV con pandas. Siempre conviene **mirar los datos** antes de hacer nada:
- `head()` muestra las primeras filas.
- `shape` dice cuántas filas y columnas hay.
- `value_counts()` dice cuántas reseñas hay de cada clase (para saber si está balanceado).

In [2]:
import pandas as pd

df_hoteles = pd.read_csv("./practica/reviews_hoteles.csv")

print("Filas y columnas:", df_hoteles.shape)
df_hoteles.head()

Filas y columnas: (40, 2)


,texto,sentimiento
0,Superó ampliamente lo que esperábamos por el p...,positivo
1,El gimnasio estaba muy bien equipado y siempre...,positivo
2,Nos sorprendió gratamente la limpieza diaria y...,positivo
3,"Ruido constante de la calle, imposible descansar",negativo
4,La pileta y el spa hicieron que la estadía fue...,positivo


In [3]:
# ¿Cuántas positivas y cuántas negativas hay?
df_hoteles['sentimiento'].value_counts()

sentimiento
positivo    20
negativo    20
Name: count, dtype: int64

Resultado: 40 reseñas, 20 positivas y 20 negativas → el dataset está **balanceado**.

Separamos el texto (lo que el modelo va a leer, `X`) de la etiqueta (lo que tiene que aprender a predecir, `y`).

In [5]:
reseñas = df_hoteles['texto']          # X: las entradas
etiquetas = df_hoteles['sentimiento']  # y: las respuestas correctas

## Paso 2 — Split train / test

Dividimos los datos en dos partes:
- **train (80%)**: con esto el modelo aprende.
- **test (20%)**: lo guardamos y **no** se lo mostramos al modelo hasta el final, para evaluar con reseñas que nunca vio.

Parámetros:
- `test_size=0.2` → 20% para test (con 40 filas: 32 train y 8 test).
- `random_state=42` → fija la aleatoriedad, así cada vez que corras obtenés la misma división.
- `stratify=etiquetas` → mantiene la proporción 50/50 de positivas y negativas en ambos conjuntos.

In [6]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    reseñas,
    etiquetas,
    test_size=0.2,
    random_state=42,
    stratify=etiquetas
)

print("Train:", len(X_train), "reseñas")
print("Test: ", len(X_test), "reseñas")
print()
print("Proporción en test:")
print(y_test.value_counts())

Train: 32 reseñas
Test:  8 reseñas

Proporción en test:
sentimiento
negativo    4
positivo    4
Name: count, dtype: int64


## Paso 3 — Vectorizar con TF-IDF

El modelo no entiende texto, solo números. `TfidfVectorizer` convierte cada reseña en un vector donde cada posición es una palabra del vocabulario y el valor indica qué tan importante es esa palabra en esa reseña.

**¿Por qué `fit_transform` en train y solo `transform` en test?**
- `fit` = **aprender** el vocabulario y los pesos IDF.
- `transform` = **usar** lo aprendido para convertir textos en vectores.

El vocabulario se tiene que aprender **solo** con train. Si hiciéramos `fit` con test, el modelo "espiaría" información de los datos de evaluación → eso es **data leakage** y daría un resultado engañosamente bueno. En la vida real, las reseñas futuras tampoco van a estar disponibles cuando entrenás.

In [7]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer()

X_train_vec = vectorizer.fit_transform(X_train)  # aprende vocabulario + transforma
X_test_vec = vectorizer.transform(X_test)        # solo transforma con el vocabulario de train

print("Forma de X_train_vec:", X_train_vec.shape)  # (reseñas, palabras del vocabulario)
print("Forma de X_test_vec: ", X_test_vec.shape)   # mismas columnas que train

Forma de X_train_vec: (32, 183)
Forma de X_test_vec:  (8, 183)


In [8]:
# Para ver un poco del vocabulario que aprendió:
print(vectorizer.get_feature_names_out()[:30])

['agua' 'al' 'alto' 'andaba' 'ascensor' 'atención' 'atendieron' 'atento'
 'avisar' 'aviso' 'ayudar' 'ayudó' 'aún' 'bien' 'buscar' 'cabellos'
 'calidad' 'calle' 'cama' 'cancelaron' 'cara' 'cargos' 'cerrada' 'check'
 'checkout' 'ciudad' 'cobrar' 'cobraron' 'comida' 'comisión']


Fijate que `X_train_vec` y `X_test_vec` tienen **la misma cantidad de columnas**: las dos usan el mismo vocabulario (el de train). Las palabras de test que no estaban en train simplemente se ignoran.

## Paso 4 — Entrenar LogisticRegression

`fit()` es donde ocurre el aprendizaje: el modelo mira los vectores de train junto con sus etiquetas y ajusta un peso para cada palabra (palabras con peso positivo empujan hacia "positivo", con peso negativo hacia "negativo").

In [9]:
from sklearn.linear_model import LogisticRegression

modelo = LogisticRegression()
modelo.fit(X_train_vec, y_train)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneOnly used for `solver` == 'sag', 'saga' or 'liblinear' to shuffle thedata. It has no effect on the other solvers.See :term:`Glossary <random_state>` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following

In [10]:
# (Opcional) ¿Qué palabras aprendió como más positivas y más negativas?
import numpy as np

palabras = vectorizer.get_feature_names_out()
pesos = modelo.coef_[0]          # un peso por palabra
orden = np.argsort(pesos)

print("Clases:", modelo.classes_)  # el peso positivo empuja hacia la 2da clase ('positivo')
print("Más negativas:", list(palabras[orden[:10]]))
print("Más positivas:", list(palabras[orden[-10:]]))

Clases: ['negativo' 'positivo']
Más negativas: ['no', 'hotel', 'ruido', 'por', 'en', 'mucho', 'del', 'de', 'eternidad', 'tardaron']
Más positivas: ['andaba', 'la', 'fue', 'nos', 'con', 'vista', 'habitación', 'todo', 'excelente', 'rapidísimo']


## Paso 5 — Calcular accuracy

Ahora sí usamos el conjunto de test. `predict()` devuelve la predicción para cada reseña y `score()` calcula el **accuracy**: el porcentaje de aciertos.

In [11]:
predicciones = modelo.predict(X_test_vec)

accuracy = modelo.score(X_test_vec, y_test)
print(f"Accuracy: {accuracy:.2f}")

Accuracy: 0.38


In [12]:
# Veamos reseña por reseña en qué acertó y en qué falló
resultado = pd.DataFrame({
    'texto': X_test.values,
    'real': y_test.values,
    'prediccion': predicciones,
})
resultado['acierto'] = resultado['real'] == resultado['prediccion']
resultado

,texto,real,prediccion,acierto
0,Reservamos una habitación con vista al mar y n...,negativo,positivo,False
1,El desayuno era muy pobre para lo que cobraban,negativo,negativo,True
2,Un hotel boutique con mucho encanto y detalles...,positivo,negativo,False
3,"Volveríamos sin dudarlo, uno de los mejores ho...",positivo,negativo,False
4,Nunca resolvieron el problema con la calefacción,negativo,positivo,False
5,El gimnasio estaba muy bien equipado y siempre...,positivo,positivo,True
6,El aire acondicionado nunca funcionó a pesar d...,negativo,positivo,False
7,Superó ampliamente lo que esperábamos por el p...,positivo,positivo,True


### ¿Por qué el accuracy da tan bajo (~0.38)?

Acertó 3 de 8: **peor que tirar una moneda** (0.50). No es un error en el código, es una limitación de los datos:

- **Muy pocos ejemplos:** el modelo aprendió con 32 reseñas. Casi cada palabra aparece una sola vez, así que no puede descubrir patrones generales.
- **Las palabras de test casi no están en train:** por ejemplo *"Nunca resolvieron el problema con la calefacción"* — "nunca", "resolvieron", "calefacción" no aparecieron en train, así que el modelo decide con palabras sueltas como "con" o "el", que no dicen nada del sentimiento.
- **Aprendió palabras sin sentido:** mirá la celda del Paso 4: entre las "más positivas" aparecen *la, fue, nos, con* y entre las "más negativas" *hotel, por, en, de*. Son palabras vacías (stopwords) que por casualidad cayeron más de un lado.
- **Test chico:** con 8 reseñas cada error mueve el accuracy un 12.5%, así que el número es muy inestable.

Conclusión: el pipeline está bien armado; lo que falta son **datos**. Con cientos o miles de reseñas este mismo código daría resultados mucho mejores.

## Paso 6 — Probar con reseñas inventadas

Para predecir textos nuevos hay que pasarlos por **el mismo** vectorizer con `transform()` (nunca `fit_transform`, porque eso crearía un vocabulario nuevo y el modelo no lo entendería).

In [ ]:
reseñas_nuevas = [
    "La habitación era espectacular y el personal muy atento, volveríamos sin dudarlo",
    "El wifi no funcionaba y la limpieza dejaba mucho que desear, un desastre",
]

reseñas_nuevas_vec = vectorizer.transform(reseñas_nuevas)  # solo transform
predicciones_nuevas = modelo.predict(reseñas_nuevas_vec)
probabilidades = modelo.predict_proba(reseñas_nuevas_vec)

for texto, pred, proba in zip(reseñas_nuevas, predicciones_nuevas, probabilidades):
    print(texto)
    print(f"  → {pred}  (prob. negativo={proba[0]:.2f}, positivo={proba[1]:.2f})")
    print()

## Conclusión

Armamos el pipeline completo: **cargar → split → TF-IDF → entrenar → evaluar → predecir**.

Cosas para tener en cuenta:
- El modelo solo "conoce" las palabras que vio en train. Si una reseña nueva usa palabras que nunca aparecieron, el modelo no tiene información para decidir.
- Con 40 reseñas el modelo es muy limitado; las probabilidades quedan cerca de 0.5 porque tiene poca evidencia. Con miles de reseñas sería mucho más seguro.
- Probá cambiar las reseñas inventadas (por ejemplo con negaciones: *"no estaba limpio"*) y fijate cómo responde.